# 영상 기반 얼굴 표정 및 감정 변화 분석

### 최종 분석 구성
- YOLO 얼굴 검출 및 얼굴별 7가지 감정 분류: **Angry, Disgust, Fear, Happy, Neutral, Sad, Surprise**
- MediaPipe Face Landmarker: 얼굴 랜드마크 및 Blendshape 특징 추출
- Happy: 일반 미소·큰 웃음·비대칭 미소 등 다양한 웃는 형태를 모두 **Happy 범주 안에서** 분석
- Angry: FACS 및 얼굴 특징을 탐색적으로 분석




(YOLO 가중치 부른 경로
from ultralytics import YOLO

face_detector = YOLO(
    "https://github.com/akanametov/yolo-face/releases/download/1.0.0/yolov11n-face.pt"
))
### 2. 감정 분류 모델

**사용 모델: EfficientNet-B0**

- 총 7개 감정 분류
  1. Neutral (중립)
  2. Happy (행복)
  3. Fear (공포)
  4. Angry (분노)
  5. Surprise (놀람)
  6. Disgust (혐오)
  7. Sad (슬픔)
- 전체 데이터: 1,096장
- Train: 761장
- Validation: 160장
- Test: 175장
- 얼굴 영역을 YOLO로 검출한 후 30% 여백을 추가해 학습 데이터를 구성함.
- 최종 선택 모델: `face_best_emotion_model.pth`
- 최고 Validation Accuracy: 90.00% (Epoch 19)
- 비교 모델: `best_emotion_model.pth` (기존 전체 이미지 모델, Test Accuracy 82.86%)

**연동 시 주의사항**
- 실제 모델 학습 시 사용한 7개 클래스 순서를 그대로 사용해야 함.
- 얼굴 모델은 얼굴 영역을 30% 여백으로 확장한 뒤 분류하도록 전처리를 맞춰야 함.
- 출력 점수는 모델의 감정별 예측 점수이며, 실제 감정의 강도나 확정적인 확률로 해석하면 안 됨.

### 3. YOLO 얼굴 검출

- YOLO 얼굴 검출 모델을 이용해 이미지에서 얼굴 영역을 추출함.
- 총 1,096장 이미지의 얼굴 검출 및 전처리를 완료함.
- 영상 적용 시에도 얼굴 검출 후 얼굴 영역을 EfficientNet에 전달해야 함.
- 학습 시 사용한 얼굴 영역 30% 여백 조건을 추론에서도 동일하게 적용해야 함.

### 4. MediaPipe 얼굴 특징 추출

MediaPipe Face Landmarker를 사용하여 얼굴 특징을 추출함.

- 얼굴 랜드마크: 478개
- Blendshape: 52개
- 주요 분석 특징:
  - 눈썹 내림: browDownLeft, browDownRight
  - 눈 찡그림: eyeSquintLeft, eyeSquintRight
  - 입술 누름: mouthPressLeft, mouthPressRight
  - 입 벌림: mouth_opening
  - 얼굴 좌우 비대칭: brow_asymmetry, mouth_asymmetry

얼굴 특징은 감정 분류 모델과 별도로 표정 패턴을 분석하는 데 사용함.

### 5. Happy 표정 분석

Happy 이미지 157장을 대상으로 얼굴 특징을 분석함.

생성한 파일:
- `happy_features.csv`
- `happy_blendshapes.csv`
- `happy_validation_30.csv`
- `happy_manual_labels.csv`

분석 방향:
- 웃음 관련 얼굴 특징 비교
- 일반적인 미소와 강한 웃음의 특징 탐색
- 좌우 비대칭 등 세부 표정 특징 탐색

주의사항:
- Happy 내부의 세부 표정 패턴은 탐색적 분석 결과임.
- 검증되지 않은 세부 표정 유형을 확정적으로 분류하지 않음.
- 모든 Happy 표정은 EfficientNet의 기본 감정 분류에서는 Happy로 표시함.

### 6. Angry 표정 분석

Angry 이미지 157장을 대상으로 MediaPipe 얼굴 특징을 추출함.

생성 파일:
- `anger_features.csv`

V_Expression 데이터 127장을 대상으로 K-means 군집분석을 수행함.

군집 결과:

| 군집 | 이미지 수 | 주요 특징 |
|---|---:|---|
| Cluster 0 | 71장 | 눈썹 내림과 눈 찡그림이 상대적으로 적음 |
| Cluster 1 | 45장 | 눈썹 내림과 눈 찡그림이 두드러짐 |
| Cluster 2 | 11장 | 입술 누름 특징이 두드러짐 |

- K-means 군집 수: 3
- 실루엣 점수: 0.2239
- 군집은 감정 강도가 아닌 얼굴 특징의 유사성에 따른 표정 패턴임.
- 약한 화남·보통 화남·강한 화남으로 해석하지 않음.
- 군집별 표본 수와 낮은 실루엣 점수를 고려해 탐색적 결과로 활용함.

영상 적용 시 Angry로 분류된 프레임에 한해 얼굴 특징을 추출하고, 저장된 K-means 모델에 입력하여 표정 패턴을 표시할 수 있음.



## 01. Google Drive 연결 및 환경 설정

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip -q install ultralytics mediapipe openpyxl

import os, re, urllib.request
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import mediapipe as mp
from PIL import Image
from ultralytics import YOLO
from torchvision import models, transforms
from google.colab import files
from IPython.display import display

PROJECT_PATH = "/content/drive/MyDrive/Face_Emotion_Project"
MODEL_DIR = os.path.join(PROJECT_PATH, "model")
FACE_DIR = os.path.join(PROJECT_PATH, "face_processed")
SPLIT_DIR = os.path.join(PROJECT_PATH, "split")
RESULT_DIR = os.path.join(PROJECT_PATH, "results")
os.makedirs(RESULT_DIR, exist_ok=True)

CLASS_NAMES = ["Angry", "Disgust", "Fear", "Happy", "Neutral", "Sad", "Surprise"]
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("Project folder:", os.path.isdir(PROJECT_PATH))

## 02. 저장된 데이터 및 모델 확인

In [ ]:
required_files = {
    "face dataset CSV": os.path.join(FACE_DIR, "face_dataset.csv"),
    "face EfficientNet": os.path.join(MODEL_DIR, "face_best_emotion_model.pth"),
    "train split": os.path.join(SPLIT_DIR, "train.csv"),
    "validation split": os.path.join(SPLIT_DIR, "validation.csv"),
    "test split": os.path.join(SPLIT_DIR, "test.csv"),
}
for name, path in required_files.items():
    print(f"{name}: {os.path.exists(path)}")

face_df = pd.read_csv(required_files["face dataset CSV"])
print("\n얼굴 이미지 데이터:", len(face_df))
print(face_df["split"].value_counts())
print("\n감정별 이미지 수:")
print(face_df["emotion"].value_counts().reindex(CLASS_NAMES))

# 기존 분할은 확인만 하며 새로 생성하거나 덮어쓰지 않습니다.
train_df = pd.read_csv(required_files["train split"])
val_df = pd.read_csv(required_files["validation split"])
test_df = pd.read_csv(required_files["test split"])
print("\n기존 분할:", len(train_df), len(val_df), len(test_df))

## 03. 얼굴 감정 분류 모델 및 YOLO 불러오기

In [ ]:
# 얼굴 crop 학습 때 사용한 평가 전처리와 동일하게 적용
EVAL_TRANSFORM = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

face_model_path = os.path.join(MODEL_DIR, "face_best_emotion_model.pth")
checkpoint = torch.load(face_model_path, map_location=DEVICE, weights_only=False)
face_model = models.efficientnet_b0(weights=None)
face_model.classifier[1] = nn.Linear(face_model.classifier[1].in_features, len(CLASS_NAMES))
face_model.load_state_dict(checkpoint["model_state_dict"])
face_model = face_model.to(DEVICE).eval()

face_detector = YOLO("https://github.com/akanametov/yolo-face/releases/download/1.0.0/yolov11n-face.pt")
print("얼굴 EfficientNet 불러오기 완료")
print("Best epoch:", checkpoint.get("epoch"))
print("Validation accuracy:", checkpoint.get("val_acc"))
print("YOLO 얼굴 검출기 불러오기 완료")

## 04. 사진 속 모든 얼굴 검출 및 감정 예측 함수

In [ ]:
def analyze_image(image_path, margin_ratio=0.30, show=True):
    """YOLO로 얼굴을 찾고, 학습과 동일한 여백·전처리로 감정을 예측합니다."""
    bgr = cv2.imread(image_path)
    if bgr is None:
        print("이미지를 읽을 수 없습니다:", image_path)
        return []

    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    detections = face_detector(rgb, verbose=False)[0]
    h, w = rgb.shape[:2]
    canvas = rgb.copy()
    outputs = []

    for box in detections.boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
        bw, bh = x2 - x1, y2 - y1
        mx, my = int(bw * margin_ratio), int(bh * margin_ratio)
        ax1, ay1 = max(0, x1-mx), max(0, y1-my)
        ax2, ay2 = min(w, x2+mx), min(h, y2+my)
        crop = rgb[ay1:ay2, ax1:ax2]
        if crop.size == 0:
            continue

        tensor = EVAL_TRANSFORM(Image.fromarray(crop)).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            probs = torch.softmax(face_model(tensor), dim=1)[0]
        idx = int(probs.argmax().item())
        score = float(probs[idx].item())
        label = f"{CLASS_NAMES[idx]} ({score*100:.1f}%)"

        cv2.rectangle(canvas, (ax1, ay1), (ax2, ay2), (0, 255, 0), 2)
        cv2.putText(canvas, label, (ax1, max(ay1-10, 22)),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 255, 0), 2)
        outputs.append({"image": os.path.basename(image_path),
                        "face_box": [ax1, ay1, ax2, ay2],
                        "emotion": CLASS_NAMES[idx],
                        "score": score,
                        "scores": {name: float(p) for name, p in zip(CLASS_NAMES, probs.cpu())}})

    if show:
        plt.figure(figsize=(10, 7))
        plt.imshow(canvas)
        plt.axis("off")
        plt.title(os.path.basename(image_path))
        plt.show()
    print("검출 얼굴 수:", len(outputs))
    for i, item in enumerate(outputs, 1):
        print(f"얼굴 {i}: {item['emotion']} ({item['score']*100:.2f}%)")
    return outputs

## 05. 새 사진 업로드 및 감정 분석 (필요할 때만 실행)

In [ ]:
uploaded = files.upload()
all_predictions = []
for image_name in uploaded.keys():
    all_predictions.extend(analyze_image(image_name))

if all_predictions:
    prediction_df = pd.DataFrame([
        {"image": x["image"], "emotion": x["emotion"], "score": x["score"]}
        for x in all_predictions
    ])
    display(prediction_df)
    prediction_df.to_csv(os.path.join(RESULT_DIR, "photo_emotion_predictions.csv"), index=False)
    print("예측 결과 저장 완료")

## 06. MediaPipe Face Landmarker 설정

In [ ]:
TASK_MODEL_PATH = "/content/face_landmarker.task"
if not os.path.exists(TASK_MODEL_PATH):
    url = "https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task"
    urllib.request.urlretrieve(url, TASK_MODEL_PATH)

BaseOptions = mp.tasks.BaseOptions
FaceLandmarker = mp.tasks.vision.FaceLandmarker
FaceLandmarkerOptions = mp.tasks.vision.FaceLandmarkerOptions
VisionRunningMode = mp.tasks.vision.RunningMode

landmarker_options = FaceLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=TASK_MODEL_PATH),
    running_mode=VisionRunningMode.IMAGE,
    num_faces=1,
    output_face_blendshapes=True
)
face_landmarker = FaceLandmarker.create_from_options(landmarker_options)
print("MediaPipe Face Landmarker 준비 완료")

## 07. 얼굴 랜드마크 및 Blendshape 특징 추출

In [ ]:
def extract_face_features(image_path):
    """한 이미지에서 정규화한 얼굴 기하 특징과 MediaPipe Blendshape를 추출합니다."""
    bgr = cv2.imread(image_path)
    if bgr is None:
        return None
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
    result = face_landmarker.detect(mp_image)
    if not result.face_landmarks:
        return None

    lm = result.face_landmarks[0]
    face_width = abs(lm[454].x - lm[234].x)
    if face_width == 0:
        return None

    row = {
        "image": os.path.basename(image_path),
        "brow_eye_distance": (abs(lm[105].y-lm[159].y) + abs(lm[334].y-lm[386].y)) / (2*face_width),
        "brow_asymmetry": abs(lm[105].y-lm[334].y) / face_width,
        "mouth_opening": abs(lm[13].y-lm[14].y) / face_width,
        "mouth_asymmetry": abs(lm[61].y-lm[291].y) / face_width,
        "mouth_width": abs(lm[61].x-lm[291].x) / face_width,
    }
    if result.face_blendshapes:
        for category in result.face_blendshapes[0]:
            row[category.category_name] = category.score
    return row

## 08. Happy 표정 분석 결과 확인

In [ ]:
# 다양한 웃는 형태는 별도 감정 클래스로 나누지 않고 Happy 안에서 분석
# 비대칭만으로 비웃음(sneer)을 확정하지 않음

RESULTS_PATH = os.path.join(PROJECT_PATH, "results")

happy_files = [
    "happy_validation_30.csv",
    "happy_manual_labels.csv",
    "happy_features.csv",
    "happy_blendshapes.csv"
]

for filename in happy_files:
    path = os.path.join(RESULTS_PATH, filename)

    if os.path.exists(path):
        data = pd.read_csv(path)
        print(f"{filename}: {data.shape[0]}행 × {data.shape[1]}열")
        display(data.head(3))
    else:
        print(f"파일 없음: {path}")

In [ ]:
# 업로드한 사진 중 첫 번째 사진의 특징 추출
feature = extract_face_features(next(iter(uploaded)))

if feature:
    display(pd.DataFrame([feature]))
else:
    print("얼굴 특징을 추출하지 못했습니다.")

## 09. Angry 표정 및 FACS 분석 결과 확인

In [ ]:
anger_path = os.path.join(PROJECT_PATH, "anger_features.csv")
if os.path.exists(anger_path):
    anger_features = pd.read_csv(anger_path)
    print("분노 특징 데이터:", anger_features.shape)
    display(anger_features.head())
else:
    print("anger_features.csv가 없습니다. 기존 원본 노트북의 분노 특징 추출 결과를 확인하세요.")

# 분노 강도 기준(0~3)
anger_level_criteria = pd.DataFrame([
    [0, "분노 관련 표정 변화 없음"],
    [1, "약한 얼굴 긴장 또는 눈썹 변화"],
    [2, "눈썹과 입 주변의 뚜렷한 긴장 변화"],
    [3, "여러 얼굴 특징에서 강한 변화"],
], columns=["Level", "기준"])
display(anger_level_criteria)
print("주의: 현재 WSEFEP의 Intensity(0~1)와 FACS AU 강도 문자는 위 0~3 라벨과 동일하지 않습니다.")
print("0~3단계 자동 분류기를 만들려면 사진별로 이 기준에 따라 별도 정답 라벨을 부여하고 검증해야 합니다.")

## 10. 분석 결과 및 파일 저장 위치

## 해석 및 사용 시 주의사항

- 얼굴 감정 모델의 출력 점수(softmax)는 감정의 실제 강도나 보정된 확률이 아닙니다.
- YOLO는 얼굴 위치를 찾고, EfficientNet은 검출된 얼굴의 7가지 감정을 분류합니다.
- Happy의 다양한 웃는 모양은 모두 Happy 범주에 포함합니다. 비대칭 미소와 비웃음은 동일한 것으로 단정하지 않습니다.
- MediaPipe 랜드마크/Blendshape 값은 표정 특징량입니다. 이 값만으로 공식 FACS Action Unit을 직접 판정했다고 표현하지 않습니다.
- 분노 0~3단계는 독립 라벨링한 자료가 있어야 자동 분류 성능을 평가할 수 있음
- 영상 프레임 처리와 시간에 따른 감정 변화 타임라인은 팀원의 영상 처리 단계에서 이 얼굴 분석 함수를 프레임별로 호출해 연결합니다.

In [ ]:
import os
import pandas as pd

PROJECT_PATH = "/content/drive/MyDrive/Face_Emotion_Project"
RESULTS_PATH = os.path.join(PROJECT_PATH, "results")

anger_df = pd.read_csv(
    os.path.join(RESULTS_PATH, "anger_features.csv")
)

print("데이터 크기:", anger_df.shape)

print("\n컬럼 목록:")
print(anger_df.columns.tolist())

print("\n데이터 앞부분:")
display(anger_df.head())

In [ ]:
import pandas as pd
from scipy.stats import spearmanr

# 분석할 주요 얼굴 특징
features = [
    "brow_eye_distance",
    "brow_asymmetry",
    "browDownLeft",
    "browDownRight",
    "eyeSquintLeft",
    "eyeSquintRight",
    "mouthPressLeft",
    "mouthPressRight",
    "mouthFrownLeft",
    "mouthFrownRight",
    "noseSneerLeft",
    "noseSneerRight",
    "mouth_opening"
]

# 숫자형 데이터로 변환
for col in features + ["Intensity"]:
    anger_df[col] = pd.to_numeric(anger_df[col], errors="coerce")

# 주요 특징 통계
stats = anger_df[features].describe().T
stats = stats[["mean", "std", "min", "max"]]

print("① 화남 표정의 주요 특징 통계")
display(stats.round(4))

# 기존 Intensity와의 상관관계
results = []

for col in features:
    temp = anger_df[[col, "Intensity"]].dropna()

    if len(temp) >= 3:
        corr, p_value = spearmanr(
            temp[col], temp["Intensity"]
        )

        results.append({
            "특징": col,
            "상관계수": corr,
            "p-value": p_value
        })

corr_df = pd.DataFrame(results)
corr_df["절댓값"] = corr_df["상관계수"].abs()
corr_df = corr_df.sort_values("절댓값", ascending=False)

print("\n② 기존 Intensity와의 상관관계")
display(corr_df.drop(columns="절댓값").round(4))

In [ ]:
import os

PROJECT_PATH = "/content/drive/MyDrive/Face_Emotion_Project"
FACE_PATH = os.path.join(PROJECT_PATH, "face_processed")

splits = ["train", "validation", "test"]

angry_images = []

for split in splits:
    folder = os.path.join(FACE_PATH, split, "Angry")

    images = [
        os.path.join(folder, f)
        for f in os.listdir(folder)
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    ]

    angry_images.extend(images)
    print(f"{split}: {len(images)}장")

print("\n전체 Angry 이미지:", len(angry_images), "장")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plot_features = [
    "browDownLeft",
    "browDownRight",
    "eyeSquintLeft",
    "eyeSquintRight",
    "mouthPressLeft",
    "mouthPressRight",
    "mouth_opening"
]

plt.figure(figsize=(12, 6))

sns.boxplot(
    data=anger_157_df[plot_features]
)

plt.title("Distribution of Angry Facial Features")
plt.ylabel("Blendshape / Feature Score")
plt.xticks(rotation=35)
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

features = [
    "browDownLeft",
    "browDownRight",
    "eyeSquintLeft",
    "eyeSquintRight",
    "mouthPressLeft",
    "mouthPressRight",
    "mouth_opening"
]

plt.figure(figsize=(10, 7))

sns.heatmap(
    anger_157_df[features].corr(method="spearman"),
    annot=True,
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
    fmt=".2f"
)

plt.title("Correlation Between Angry Facial Features")
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

cluster_features = [
    "browDownLeft",
    "browDownRight",
    "eyeSquintLeft",
    "eyeSquintRight",
    "mouthPressLeft",
    "mouthPressRight",
    "mouth_opening"
]

X = anger_157_df[cluster_features].copy()

# 표준화
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 군집 수별 실루엣 점수 확인
for k in range(2, 6):
    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=20
    )

    labels = kmeans.fit_predict(X_scaled)

    score = silhouette_score(X_scaled, labels)

    print(f"군집 수 {k}: 실루엣 점수 {score:.4f}")

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import pandas as pd

cluster_features = [
    "browDownLeft",
    "browDownRight",
    "eyeSquintLeft",
    "eyeSquintRight",
    "mouthPressLeft",
    "mouthPressRight",
    "mouth_opening"
]

# 데이터 표준화
X = anger_157_df[cluster_features].copy()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# K-means 군집 분석
kmeans = KMeans(
    n_clusters=2,
    random_state=42,
    n_init=20
)

anger_157_df["cluster"] = kmeans.fit_predict(X_scaled)

# 군집별 특징 평균
cluster_result = anger_157_df.groupby("cluster")[cluster_features].mean()

print("군집별 얼굴 특징 평균")
display(cluster_result.round(4))

print("\n군집별 사진 수")
print(anger_157_df["cluster"].value_counts().sort_index())

In [ ]:
import matplotlib.pyplot as plt
import os
from PIL import Image

fig, axes = plt.subplots(2, 5, figsize=(15, 6))

for cluster_id in [0, 1]:
    samples = anger_157_df[
        anger_157_df["cluster"] == cluster_id
    ].sample(
        n=5,
        random_state=42
    )

    for j, (_, row) in enumerate(samples.iterrows()):
        image_path = os.path.join(
            PROJECT_PATH,
            "face_processed",
            row["split"],
            "Angry",
            row["image"]
        )

        img = Image.open(image_path)

        axes[cluster_id, j].imshow(img)
        axes[cluster_id, j].axis("off")

        if j == 0:
            axes[cluster_id, j].set_title(
                f"Cluster {cluster_id}"
            )

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

cluster_means = anger_157_df.groupby("cluster")[
    cluster_features
].mean()

cluster_means.T.plot(
    kind="bar",
    figsize=(12, 6),
    width=0.8
)

plt.title("Comparison of Angry Expression Clusters")
plt.ylabel("Average Feature Score")
plt.xlabel("Facial Features")
plt.xticks(rotation=35)
plt.legend(title="Cluster")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
from scipy.stats import mannwhitneyu
import pandas as pd

features = [
    "browDownLeft",
    "browDownRight",
    "eyeSquintLeft",
    "eyeSquintRight",
    "mouthPressLeft",
    "mouthPressRight",
    "mouth_opening"
]

results = []

for feature in features:
    group0 = anger_157_df.loc[
        anger_157_df["cluster"] == 0, feature
    ]
    group1 = anger_157_df.loc[
        anger_157_df["cluster"] == 1, feature
    ]

    statistic, p_value = mannwhitneyu(
        group0, group1, alternative="two-sided"
    )

    results.append({
        "특징": feature,
        "Cluster 0 평균": group0.mean(),
        "Cluster 1 평균": group1.mean(),
        "p-value": p_value
    })

result_df = pd.DataFrame(results)

display(result_df.round(5))

In [ ]:
# 파일명으로 데이터셋 출처 구분
anger_157_df["source"] = anger_157_df["image"].apply(
    lambda x: "V_Expression" if x.startswith("VEXP_")
    else "WSEFEP"
)

# 출처별 사진 수
print("데이터셋별 사진 수")
print(anger_157_df["source"].value_counts())

# 군집별 데이터셋 출처 분포
print("\n군집별 데이터셋 출처")
display(
    pd.crosstab(
        anger_157_df["cluster"],
        anger_157_df["source"]
    )
)

In [ ]:
# V_Expression 데이터만 추출
v_angry = anger_157_df[
    anger_157_df["source"] == "V_Expression"
].copy()

# 군집 분석에 사용할 특징
cluster_features = [
    "browDownLeft",
    "browDownRight",
    "eyeSquintLeft",
    "eyeSquintRight",
    "mouthPressLeft",
    "mouthPressRight",
    "mouth_opening"
]

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

X = v_angry[cluster_features]

scaler_v = StandardScaler()
X_scaled_v = scaler_v.fit_transform(X)

for k in range(2, 6):
    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=20
    )

    labels = model.fit_predict(X_scaled_v)

    score = silhouette_score(X_scaled_v, labels)

    print(f"군집 수 {k}: 실루엣 점수 {score:.4f}")

In [ ]:
# V_Expression 3개 군집 분석
kmeans_v = KMeans(
    n_clusters=3,
    random_state=42,
    n_init=20
)

v_angry["cluster"] = kmeans_v.fit_predict(X_scaled_v)

# 군집별 특징 평균
cluster_v_result = v_angry.groupby("cluster")[
    cluster_features
].mean()

print("V_Expression 군집별 특징 평균")
display(cluster_v_result.round(4))

print("\n군집별 사진 수")
print(v_angry["cluster"].value_counts().sort_index())

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import os

fig, axes = plt.subplots(3, 5, figsize=(15, 9))

for cluster_id in range(3):
    samples = v_angry[
        v_angry["cluster"] == cluster_id
    ].sample(
        n=min(5, len(v_angry[v_angry["cluster"] == cluster_id])),
        random_state=42
    )

    for j, (_, row) in enumerate(samples.iterrows()):
        image_path = os.path.join(
            PROJECT_PATH,
            "face_processed",
            row["split"],
            "Angry",
            row["image"]
        )

        img = Image.open(image_path)

        axes[cluster_id, j].imshow(img)
        axes[cluster_id, j].axis("off")

        if j == 0:
            axes[cluster_id, j].set_title(
                f"Cluster {cluster_id}"
            )

plt.tight_layout()
plt.show()

In [ ]:
# Cluster 2의 입술 압박 특징 개별 확인

cluster2 = v_angry[v_angry["cluster"] == 2].copy()

check_columns = [
    "image",
    "mouthPressLeft",
    "mouthPressRight",
    "browDownLeft",
    "browDownRight",
    "eyeSquintLeft",
    "eyeSquintRight",
    "mouth_opening"
]

print("Cluster 2 사진 수:", len(cluster2))
display(
    cluster2[check_columns].sort_values(
        by="mouthPressLeft",
        ascending=False
    ).round(4)
)

print("\n입술 압박 특징 요약")
display(
    cluster2[["mouthPressLeft", "mouthPressRight"]]
    .describe()
    .round(4)
)

In [ ]:
# 세 군집의 입술 압박 차이 비교

print(
    v_angry.groupby("cluster")[
        ["mouthPressLeft", "mouthPressRight"]
    ].agg(["mean", "median", "std"]).round(4)
)

In [ ]:
from scipy.stats import kruskal

groups_left = [
    v_angry[v_angry["cluster"] == i]["mouthPressLeft"]
    for i in range(3)
]

groups_right = [
    v_angry[v_angry["cluster"] == i]["mouthPressRight"]
    for i in range(3)
]

result_left = kruskal(*groups_left)
result_right = kruskal(*groups_right)

print("왼쪽 입술 압박")
print(f"H-statistic: {result_left.statistic:.4f}")
print(f"p-value: {result_left.pvalue:.6f}")

print("\n오른쪽 입술 압박")
print(f"H-statistic: {result_right.statistic:.4f}")
print(f"p-value: {result_right.pvalue:.6f}")

In [ ]:
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests
from itertools import combinations
import pandas as pd

features = ["mouthPressLeft", "mouthPressRight"]

for feature in features:
    results = []

    for a, b in combinations([0, 1, 2], 2):
        group_a = v_angry[v_angry["cluster"] == a][feature]
        group_b = v_angry[v_angry["cluster"] == b][feature]

        stat, p = mannwhitneyu(
            group_a, group_b,
            alternative="two-sided"
        )

        results.append({
            "비교": f"Cluster {a} vs Cluster {b}",
            "p-value": p
        })

    # 세 번의 비교에 대한 Holm 보정
    adjusted = multipletests(
        [r["p-value"] for r in results],
        method="holm"
    )[1]

    for i in range(len(results)):
        results[i]["보정 p-value"] = adjusted[i]
        results[i]["유의 여부"] = (
            "유의함" if adjusted[i] < 0.05 else "유의하지 않음"
        )

    print(f"\n[{feature}]")
    display(pd.DataFrame(results).round(6))

In [ ]:
from scipy.stats import kruskal, mannwhitneyu
from statsmodels.stats.multitest import multipletests
from itertools import combinations
import pandas as pd

features = [
    "browDownLeft",
    "browDownRight",
    "eyeSquintLeft",
    "eyeSquintRight"
]

all_results = []

for feature in features:
    groups = [
        v_angry[v_angry["cluster"] == i][feature]
        for i in range(3)
    ]

    h_stat, kw_p = kruskal(*groups)

    for a, b in combinations([0, 1, 2], 2):
        group_a = v_angry[v_angry["cluster"] == a][feature]
        group_b = v_angry[v_angry["cluster"] == b][feature]

        u_stat, p = mannwhitneyu(
            group_a, group_b,
            alternative="two-sided"
        )

        all_results.append({
            "특징": feature,
            "검정": f"Cluster {a} vs Cluster {b}",
            "Kruskal p-value": kw_p,
            "사후검정 p-value": p
        })

# 전체 4개 특징 × 3개 비교 = 12개 검정에 Holm 보정
adjusted_p = multipletests(
    [r["사후검정 p-value"] for r in all_results],
    method="holm"
)[1]

for i, p in enumerate(adjusted_p):
    all_results[i]["Holm 보정 p-value"] = p
    all_results[i]["유의 여부"] = (
        "유의함" if p < 0.05 else "유의하지 않음"
    )

display(pd.DataFrame(all_results).round(6))

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# 분석할 7개 얼굴 특징
features = [
    "browDownLeft",
    "browDownRight",
    "eyeSquintLeft",
    "eyeSquintRight",
    "mouthPressLeft",
    "mouthPressRight",
    "mouth_opening"
]

# 군집별 평균 계산
cluster_means = (
    v_angry.groupby("cluster")[features]
    .mean()
)

# 군집별 사진 수
cluster_counts = v_angry["cluster"].value_counts().sort_index()

print("군집별 사진 수")
print(cluster_counts)

# 특징별 표준화 (각 특징의 군집 간 상대적 차이)
heatmap_data = (
    cluster_means - cluster_means.mean(axis=0)
) / cluster_means.std(axis=0)

# 히트맵
fig, ax = plt.subplots(figsize=(12, 5))

im = ax.imshow(
    heatmap_data.T,
    cmap="RdBu_r",
    aspect="auto",
    vmin=-1.2,
    vmax=1.2
)

ax.set_xticks(range(3))
ax.set_xticklabels([
    f"Cluster {i}\n(n={cluster_counts[i]})"
    for i in range(3)
])

ax.set_yticks(range(len(features)))
ax.set_yticklabels(features)

# 각 칸에 표준화 점수 표시
for i in range(len(features)):
    for j in range(3):
        value = heatmap_data.iloc[j, i]
        ax.text(
            j, i,
            f"{value:.2f}",
            ha="center",
            va="center",
            color="black"
        )

ax.set_title(
    "Angry Expression Clustering - Facial Feature Heatmap",
    fontsize=14,
    pad=15
)

ax.set_xlabel("Cluster")
ax.set_ylabel("Facial Features")

fig.colorbar(
    im,
    ax=ax,
    label="Standardized Mean (Z-score)"
)

plt.tight_layout()
plt.show()

In [ ]:
print(anger_df.columns.tolist())
print(anger_df.head())

In [ ]:
# 현재 노트북에 만들어진 변수 확인
for name in globals():
    if any(word in name.lower() for word in ["kmeans", "scaler", "cluster"]):
        print(name, type(globals()[name]))

In [ ]:
import os
import joblib

PROJECT_PATH = "/content/drive/MyDrive/Face_Emotion_Project"
RESULTS_PATH = os.path.join(PROJECT_PATH, "results")

os.makedirs(RESULTS_PATH, exist_ok=True)

joblib.dump(
    scaler_v,
    os.path.join(RESULTS_PATH, "angry_scaler.pkl")
)

joblib.dump(
    kmeans_v,
    os.path.join(RESULTS_PATH, "angry_kmeans.pkl")
)

print("Angry K-means 모델 저장 완료!")

In [ ]:
import os

for root, dirs, files in os.walk("/content/drive/MyDrive/Face_Emotion_Project"):
    for file in files:
        if file == "face_landmarker.task":
            print(os.path.join(root, file))

In [ ]:
import os
import urllib.request

PROJECT_PATH = "/content/drive/MyDrive/Face_Emotion_Project"
MODEL_PATH = os.path.join(PROJECT_PATH, "model")

os.makedirs(MODEL_PATH, exist_ok=True)

url = "https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task"

save_path = os.path.join(MODEL_PATH, "face_landmarker.task")

urllib.request.urlretrieve(url, save_path)

print("저장 완료:", save_path)
print("파일 크기:", round(os.path.getsize(save_path) / 1024 / 1024, 2), "MB")

In [ ]:
import os

PROJECT_PATH = "/content/drive/MyDrive/Face_Emotion_Project"
MODEL_PATH = os.path.join(PROJECT_PATH, "model")

files = [
    "angry_kmeans.pkl",
    "angry_scaler.pkl",
    "face_best_emotion_model.pth",
    "face_landmarker.task"
]

for filename in files:
    path = os.path.join(MODEL_PATH, filename)

    if os.path.exists(path):
        print(f"✅ {filename} : 존재함 ({os.path.getsize(path):,} bytes)")
    else:
        print(f"❌ {filename} : 없음")

In [ ]:
import joblib
import os

MODEL_PATH = "/content/drive/MyDrive/Face_Emotion_Project/model"

# 저장된 화남 분석 모델 불러오기
angry_kmeans = joblib.load(
    os.path.join(MODEL_PATH, "angry_kmeans.pkl")
)

angry_scaler = joblib.load(
    os.path.join(MODEL_PATH, "angry_scaler.pkl")
)

print("K-means 모델:", angry_kmeans)
print("Scaler 모델:", angry_scaler)

print("\n군집 개수:", angry_kmeans.n_clusters)
print("입력 특징 개수:", angry_kmeans.n_features_in_)
print("Scaler 입력 특징 개수:", angry_scaler.n_features_in_)

In [ ]:
print("화남 군집 분석에 사용한 특징 목록:")
print(cluster_features)

print("\n특징 개수:", len(cluster_features))

In [ ]:
import json
import os

MODEL_PATH = "/content/drive/MyDrive/Face_Emotion_Project/model"

feature_path = os.path.join(
    MODEL_PATH,
    "angry_cluster_features.json"
)

with open(feature_path, "w", encoding="utf-8") as f:
    json.dump(cluster_features, f, ensure_ascii=False, indent=2)

print("저장 완료:", feature_path)

In [ ]:
import pandas as pd
import os

RESULTS_PATH = "/content/drive/MyDrive/Face_Emotion_Project/results"

anger_df = pd.read_csv(
    os.path.join(RESULTS_PATH, "anger_features.csv")
)

print("CSV 크기:", anger_df.shape)
print("\n특징값 결측치:")
print(anger_df[cluster_features].isnull().sum())

print("\n특징값 일부:")
print(anger_df[cluster_features].head())

In [ ]:
print("이미지 파일명 앞부분:")
print(anger_df["image"].head(20).to_list())

print("\n전체 이미지 파일명 개수:", anger_df["image"].nunique())

In [ ]:
# 데이터 출처별 파일명 개수 확인
anger_df["source"] = anger_df["image"].apply(
    lambda x: "WSEFEP" if str(x).startswith("WSEFEP_") else "V_Expression"
)

print(anger_df["source"].value_counts())

print("\nV_Expression 파일명 예시:")
print(
    anger_df.loc[
        anger_df["source"] == "V_Expression", "image"
    ].head(20).to_list()
)

In [ ]:
# V_Expression 데이터만 선택
vexp_df = anger_df[
    anger_df["source"] == "V_Expression"
].copy()

# 학습할 때 사용한 7개 특징만 추출
X = vexp_df[cluster_features]

# 저장된 Scaler로 표준화
X_scaled = angry_scaler.transform(X)

# 저장된 K-means로 군집 예측
vexp_df["predicted_cluster"] = angry_kmeans.predict(X_scaled)

print("예측 데이터 수:", len(vexp_df))
print("\n군집별 이미지 개수:")
print(vexp_df["predicted_cluster"].value_counts().sort_index())

print("\n군집별 특징 평균:")
print(
    vexp_df.groupby("predicted_cluster")[cluster_features].mean().round(4)
)

In [ ]:
import os

save_path = os.path.join(
    RESULTS_PATH,
    "angry_cluster_validation.csv"
)

vexp_df.to_csv(save_path, index=False, encoding="utf-8-sig")

print("저장 완료:", save_path)

In [ ]:
save_path = os.path.join(
    RESULTS_PATH,
    "angry_cluster_validation.csv"
)

vexp_df.to_csv(save_path, index=False, encoding="utf-8-sig")

print("저장 완료:", save_path)
print("저장 데이터:", vexp_df.shape)

In [ ]:
import pandas as pd
import os

RESULTS_PATH = "/content/drive/MyDrive/Face_Emotion_Project/results"

happy_features_df = pd.read_csv(
    os.path.join(RESULTS_PATH, "happy_features.csv")
)

happy_blendshapes_df = pd.read_csv(
    os.path.join(RESULTS_PATH, "happy_blendshapes.csv")
)

print("1. Happy Features")
print("크기:", happy_features_df.shape)
print("컬럼:", happy_features_df.columns.tolist())

print("\n2. Happy Blendshapes")
print("크기:", happy_blendshapes_df.shape)
print("컬럼:", happy_blendshapes_df.columns.tolist())

print("\n3. Happy Features 미리보기")
display(happy_features_df.head())

In [ ]:
# 행복 데이터 결측치 확인
print("1. Happy Features 결측치")
print(happy_features_df.isnull().sum())

print("\n2. Happy Blendshapes 결측치")
print(happy_blendshapes_df.isnull().sum())

# 주요 특징 통계
print("\n3. Happy Features 기술통계")
display(
    happy_features_df[
        ["mouth_asymmetry", "mouth_opening", "mouth_width"]
    ].describe().round(4)
)

# 주요 행복 관련 Blendshapes 통계
happy_key_features = [
    "mouthSmileLeft",
    "mouthSmileRight",
    "cheekSquintLeft",
    "cheekSquintRight",
    "eyeSquintLeft",
    "eyeSquintRight",
    "mouthStretchLeft",
    "mouthStretchRight"
]

print("\n4. 주요 Blendshapes 기술통계")
display(
    happy_blendshapes_df[happy_key_features]
    .describe()
    .round(4)
)

In [ ]:
import numpy as np
from scipy.stats import spearmanr

# 좌우 평균값 계산
happy_blendshapes_df["mouthSmile_mean"] = (
    happy_blendshapes_df["mouthSmileLeft"] +
    happy_blendshapes_df["mouthSmileRight"]
) / 2

happy_blendshapes_df["eyeSquint_mean"] = (
    happy_blendshapes_df["eyeSquintLeft"] +
    happy_blendshapes_df["eyeSquintRight"]
) / 2

# 상관관계 분석
features_to_check = [
    "mouthSmile_mean",
    "eyeSquint_mean"
]

corr, p_value = spearmanr(
    happy_blendshapes_df["mouthSmile_mean"],
    happy_blendshapes_df["eyeSquint_mean"]
)

print("입꼬리 움직임과 눈 찡그림의 상관관계")
print(f"Spearman 상관계수: {corr:.4f}")
print(f"p-value: {p_value:.6f}")

if p_value < 0.05:
    print("통계적으로 유의한 상관관계가 관찰됨")
else:
    print("통계적으로 유의한 상관관계가 확인되지 않음")

In [ ]:
from scipy.stats import wilcoxon

# 좌우 입꼬리 차이
left_smile = happy_blendshapes_df["mouthSmileLeft"]
right_smile = happy_blendshapes_df["mouthSmileRight"]

# 좌우 차이 계산
smile_diff = left_smile - right_smile

print("왼쪽 입꼬리 평균:", round(left_smile.mean(), 4))
print("오른쪽 입꼬리 평균:", round(right_smile.mean(), 4))
print("좌우 차이 평균:", round(smile_diff.mean(), 4))

# 좌우 차이 통계 검정
stat, p = wilcoxon(left_smile, right_smile)

print("\nWilcoxon 검정")
print("통계량:", round(stat, 4))
print("p-value:", p)

if p < 0.05:
    print("좌우 입꼬리 움직임에 통계적으로 유의한 차이가 있음")
else:
    print("좌우 입꼬리 움직임에 통계적으로 유의한 차이가 확인되지 않음")

In [ ]:
print(
    happy_features_df["filename"].equals(
        happy_blendshapes_df["filename"]
    )
)

In [ ]:
from scipy.stats import spearmanr

corr, p_value = spearmanr(
    happy_blendshapes_df["mouthSmile_mean"],
    happy_features_df["mouth_opening"]
)

print("입꼬리 움직임과 입 벌림 정도의 관계")
print(f"Spearman 상관계수: {corr:.4f}")
print(f"p-value: {p_value:.8f}")

In [ ]:
import os
import pandas as pd

RESULTS_PATH = "/content/drive/MyDrive/Face_Emotion_Project/results"

# 파일명 일치 여부 확인
same_order = happy_features_df["filename"].equals(
    happy_blendshapes_df["filename"]
)

print("두 CSV 이미지 순서 일치:", same_order)

if same_order:
    # 행복 분석 결과 정리
    happy_summary = pd.DataFrame({
        "analysis": [
            "입꼬리와 눈 찡그림 상관관계",
            "좌우 입꼬리 차이",
            "입꼬리와 입 벌림 상관관계"
        ],
        "method": [
            "Spearman",
            "Wilcoxon",
            "Spearman"
        ],
        "statistic": [
            0.4262,
            3756.0,
            0.3961
        ],
        "p_value": [
            0.00000028,
            0.000018202,
            0.00000028
        ],
        "sample_count": [157, 157, 157]
    })

    save_path = os.path.join(
        RESULTS_PATH,
        "happy_analysis_summary.csv"
    )

    happy_summary.to_csv(
        save_path,
        index=False,
        encoding="utf-8-sig"
    )

    print("\n행복 분석 결과 저장 완료!")
    print(save_path)
    display(happy_summary)

else:
    print("이미지 순서가 달라. 파일명 기준으로 다시 연결해야 해.")

In [ ]:
import cv2

video_path = "/content/8724306-uhd_2160_4096_25fps.mp4"

cap = cv2.VideoCapture(video_path)

fps = cap.get(cv2.CAP_PROP_FPS)
frame_count = cap.get(cv2.CAP_PROP_FRAME_COUNT)

print("FPS:", fps)
print("전체 프레임:", frame_count)
print("영상 길이:", round(frame_count / fps, 2), "초")

cap.release()

In [ ]:
video_path = "/content/drive/MyDrive/Face_Emotion_Project/8724306-uhd_2160_4096_25fps.mp4"

In [ ]:
import cv2
import os
import matplotlib.pyplot as plt
from ultralytics import YOLO

# 1. 영상 경로
video_path = "/content/drive/MyDrive/Face_Emotion_Project/8724306-uhd_2160_4096_25fps.mp4"

# 2. 영상 열기
cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    raise Exception("영상을 열 수 없습니다. 경로 또는 코덱을 확인하세요.")

fps = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

print("영상 FPS:", fps)
print("전체 프레임:", total_frames)
print("영상 길이:", round(total_frames / fps, 2), "초")

# 3. YOLO 얼굴 감지 모델 불러오기
face_detector = YOLO(
    "https://github.com/akanametov/yolo-face/releases/download/1.0.0/yolov11n-face.pt"
)

# 4. 1초 간격으로 프레임 추출 및 얼굴 감지
results_list = []

for sec in range(0, int(total_frames / fps) + 1):
    cap.set(cv2.CAP_PROP_POS_MSEC, sec * 1000)
    ret, frame = cap.read()

    if not ret:
        continue

    results = face_detector(frame, verbose=False)

    boxes = results[0].boxes

    face_count = len(boxes)

    results_list.append((sec, frame, results[0], face_count))

    print(f"{sec}초: 얼굴 {face_count}개 감지")

cap.release()

# 5. 첫 번째 프레임 결과 시각화
if results_list:
    sec, frame, result, count = results_list[0]

    annotated = result.plot()

    plt.figure(figsize=(10, 6))
    plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    plt.title(f"{sec} second / Faces: {count}")
    plt.axis("off")
    plt.show()
else:
    print("추출된 프레임이 없습니다.")

In [ ]:
import torch
import os

model_path = "/content/drive/MyDrive/Face_Emotion_Project/model/face_best_emotion_model.pth"

checkpoint = torch.load(
    model_path,
    map_location="cpu",
    weights_only=True
)

print("저장된 객체 유형:", type(checkpoint))

if isinstance(checkpoint, dict):
    print("저장된 키:", list(checkpoint.keys())[:20])

In [ ]:
print("감정 클래스 순서:")
print(checkpoint["class_names"])

print("\n학습 Epoch:")
print(checkpoint["epoch"])

print("\n검증 정확도:")
print(checkpoint["val_acc"])

In [ ]:
import torch
import torch.nn as nn
from torchvision import models

# 실행 장치
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("실행 장치:", device)

# EfficientNet-B0 모델 생성
emotion_model = models.efficientnet_b0(weights=None)

# 마지막 분류층을 7개 감정으로 변경
emotion_model.classifier[1] = nn.Linear(
    emotion_model.classifier[1].in_features,
    7
)

# 학습된 가중치 적용
emotion_model.load_state_dict(checkpoint["model_state_dict"])

emotion_model = emotion_model.to(device)
emotion_model.eval()

print("EfficientNet 모델 로딩 성공!")
print("분류 클래스:", checkpoint["class_names"])

In [ ]:
import cv2
import os
import torch
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

# 영상 경로
video_path = "/content/drive/MyDrive/Face_Emotion_Project/8724306-uhd_2160_4096_25fps.mp4"

# 결과 저장 경로
RESULT_DIR = "/content/drive/MyDrive/Face_Emotion_Project/results"
os.makedirs(RESULT_DIR, exist_ok=True)

# 영상 열기
cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    raise Exception("영상을 열 수 없습니다.")

fps = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
duration = total_frames / fps

video_results = []
annotated_frames = []

# 1초 간격으로 분석
for sec in range(int(duration)):

    cap.set(cv2.CAP_PROP_POS_MSEC, sec * 1000)
    ret, frame = cap.read()

    if not ret:
        continue

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    # YOLO 얼굴 검출
    detections = face_detector(rgb, verbose=False)[0]

    h, w = rgb.shape[:2]
    canvas = rgb.copy()

    for box in detections.boxes:

        x1, y1, x2, y2 = map(
            int, box.xyxy[0].tolist()
        )

        bw = x2 - x1
        bh = y2 - y1

        # 학습 당시와 동일한 30% 여백
        mx = int(bw * 0.30)
        my = int(bh * 0.30)

        ax1 = max(0, x1 - mx)
        ay1 = max(0, y1 - my)
        ax2 = min(w, x2 + mx)
        ay2 = min(h, y2 + my)

        crop = rgb[ay1:ay2, ax1:ax2]

        if crop.size == 0:
            continue

        # 학습 당시와 동일한 전처리
        tensor = EVAL_TRANSFORM(
            Image.fromarray(crop)
        ).unsqueeze(0).to(DEVICE)

        # 감정 예측
        with torch.no_grad():
            probs = torch.softmax(
                face_model(tensor), dim=1
            )[0]

        idx = int(probs.argmax().item())
        score = float(probs[idx].item())
        emotion = CLASS_NAMES[idx]

        # 화면에 감정 표시
        label = f"{emotion} ({score*100:.1f}%)"

        cv2.rectangle(
            canvas,
            (ax1, ay1),
            (ax2, ay2),
            (0, 255, 0),
            2
        )

        cv2.putText(
            canvas,
            label,
            (ax1, max(ay1 - 10, 25)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (0, 255, 0),
            2
        )

        # 결과 저장
        video_results.append({
            "time_sec": sec,
            "emotion": emotion,
            "score": score
        })

    annotated_frames.append((sec, canvas))

    print(f"{sec}초 분석 완료")

cap.release()

# 결과 DataFrame
video_df = pd.DataFrame(video_results)

print("\n===== 영상 감정 분석 결과 =====")
display(video_df)

# CSV 저장
save_path = os.path.join(
    RESULT_DIR,
    "video_emotion_predictions.csv"
)

video_df.to_csv(
    save_path,
    index=False,
    encoding="utf-8-sig"
)

print("결과 저장 완료:", save_path)

# 첫 번째 프레임 시각화
if annotated_frames:
    sec, image = annotated_frames[0]

    plt.figure(figsize=(8, 6))
    plt.imshow(image)
    plt.title(f"Video Emotion Analysis - {sec} sec")
    plt.axis("off")
    plt.show()

# 감정별 예측 횟수
if not video_df.empty:
    print("\n===== 감정별 예측 횟수 =====")
    display(
        video_df["emotion"]
        .value_counts()
        .to_frame("count")
    )